# Qlib Quickstart
This notebook demonstrates the DataPipeline with yfinance fallback.

The `DataPipeline` class provides a unified interface for fetching market data.
When Qlib is not installed or data is unavailable, it automatically falls back
to yfinance for OHLCV data retrieval.


In [ ]:
import sys
from pathlib import Path

# Add project root to path so src/ imports work
repo_root = Path('.').resolve().parent
sys.path.insert(0, str(repo_root))
print(f'Project root: {repo_root}')


In [ ]:
# Initialize DataPipeline and fetch OHLCV data via yfinance fallback
from src.utils.config import get_config
from src.core.data_pipeline import DataPipeline

config = get_config()
pipeline = DataPipeline(config)

try:
    df = pipeline.yfinance_fallback(
        tickers=['SPY', 'AAPL', 'MSFT'],
        start='2023-01-01',
        end='2023-03-31',
    )
    print(f'Fetched {len(df)} rows x {len(df.columns)} columns')
    print(f'Tickers: {df["ticker"].unique().tolist()}')
except Exception as e:
    print(f'Data fetch error (no network?): {e}')
    import pandas as pd
    import numpy as np
    dates = pd.bdate_range('2023-01-01', periods=60)
    df = pd.DataFrame({
        'close': np.random.uniform(100, 200, len(dates)),
        'open': np.random.uniform(99, 199, len(dates)),
        'high': np.random.uniform(101, 201, len(dates)),
        'low': np.random.uniform(98, 198, len(dates)),
        'volume': np.random.randint(1_000_000, 10_000_000, len(dates)),
        'ticker': 'SPY',
    }, index=dates)
    print(f'Using synthetic data: {len(df)} rows')


In [ ]:
# Display the OHLCV DataFrame
print('OHLCV Data (first 5 rows):')
print(df.head())
print(f'\nDate range: {df.index.min()} to {df.index.max()}')
print(f'Columns: {df.columns.tolist()}')


In [ ]:
# Compute simple features (returns, momentum)
import pandas as pd

feature_frames = []
for ticker, grp in df.groupby('ticker'):
    grp = grp.sort_index()
    feat = pd.DataFrame(index=grp.index)
    feat['close'] = grp['close']
    feat['returns'] = grp['close'].pct_change(1)
    feat['mom5'] = grp['close'].pct_change(5)
    feat['mom20'] = grp['close'].pct_change(20)
    feat['vol_ratio'] = grp['volume'] / grp['volume'].rolling(20).mean()
    feat['ticker'] = ticker
    feature_frames.append(feat)

features = pd.concat(feature_frames).dropna()
print(f'Feature matrix: {features.shape}')
print(features[['returns', 'mom5', 'mom20', 'vol_ratio']].describe().round(4))


In [ ]:
# Try to load the factor library and create an augmented dataset
try:
    lib_df = pipeline.load_factor_library()
    if lib_df.empty:
        print('Factor library is empty or not found — run RDAgentRunner first.')
    else:
        print(f'Factor library loaded: {lib_df.shape}')
        print(lib_df.head())
except Exception as e:
    print(f'Factor library load error: {e}')


## Summary

The `DataPipeline` provides a unified interface for market data:

- `yfinance_fallback()` — fetches OHLCV from Yahoo Finance when Qlib data is unavailable
- `get_features()` — returns Qlib Alpha158 features, falling back to yfinance
- `load_factor_library()` — loads RD-Agent discovered factors from `outputs/factor_library.json`
- `create_dataset_with_custom_factors()` — merges Qlib features with custom factors

Run `aiquant setup` to initialize directories before using the pipeline in production.
